# Tutorial 4: Running CAPO Training Experiments

This tutorial walks through setting up and running CAPO training experiments using the vendored VERL framework.

## Prerequisites

- CAPO package installed (`pip install -e .`)
- Vendored VERL available (included in `src/capo/experiments/verl/`)
- Ray cluster available (or local mode for small experiments)
- GPU(s) for actual training (CPU-only for data prep)

## Overview

The CAPO training pipeline has three main stages:

1. **Data Preparation**: Convert task data to VERL-compatible parquet format
2. **Configuration**: Set up Hydra configs for the experiment
3. **Training**: Run the PPO trainer with CAPO advantage estimation

In [ ]:
# Check environment
import sys
print(f"Python: {sys.executable}")

# Verify CAPO package
import capo
print(f"CAPO version: {capo.__version__}")

: 

## 1. Data Preparation

CAPO experiments use VERL's `RLHFDataset` format. We provide a script to prepare the CountDown dataset.

In [ ]:
from pathlib import Path
import tempfile

# Create output directory for our experiment
DATA_DIR = Path(tempfile.mkdtemp(prefix="capo_tutorial_"))
print(f"Data directory: {DATA_DIR}")

In [ ]:
# Prepare a small synthetic dataset (no HuggingFace download needed)
# For real experiments, you'd use the full CountDown dataset

import pandas as pd
import json

def build_countdown_prompt(target: int, numbers: list) -> str:
    """Build a CountDown task prompt."""
    nums = ", ".join(map(str, numbers))
    return (
        "You are given a target and a multiset of numbers. "
        "Construct a single arithmetic expression using each number exactly once "
        "and using only +, -, *, /, and parentheses so that the expression evaluates "
        f"to the target.\n\nTarget: {target}\nNumbers: {nums}\n\n"
        "Format your response as:\n"
        "<think>...optional reasoning...</think>\n"
        "<answer>EXPRESSION</answer>\n\n"
        "The expression in <answer> must use each number exactly once."
    )

def create_verl_row(target: int, numbers: list, idx: int) -> dict:
    """Create a VERL-compatible data row."""
    return {
        "prompt": [{"role": "user", "content": build_countdown_prompt(target, numbers)}],
        "data_source": "countdown",
        "reward_model": {
            "ground_truth": {"target": target, "numbers": numbers}
        },
        "extra_info": {"index": idx},
    }

# Generate some simple synthetic examples
import random
random.seed(42)

train_rows = []
for i in range(100):  # Small training set
    nums = [random.randint(1, 10) for _ in range(3)]
    target = sum(nums)  # Simple: target = sum (always solvable with +)
    train_rows.append(create_verl_row(target, nums, i))

test_rows = []
for i in range(20):  # Small test set
    nums = [random.randint(1, 10) for _ in range(3)]
    target = sum(nums)
    test_rows.append(create_verl_row(target, nums, i))

# Save as parquet
train_df = pd.DataFrame(train_rows)
test_df = pd.DataFrame(test_rows)

train_df.to_parquet(DATA_DIR / "train.parquet", index=False)
test_df.to_parquet(DATA_DIR / "test.parquet", index=False)

print(f"Created {len(train_df)} training examples")
print(f"Created {len(test_df)} test examples")
print(f"\nFiles written to: {DATA_DIR}")

In [ ]:
# Inspect an example
example = train_df.iloc[0]
print("Example row:")
print(f"  Prompt: {example['prompt'][0]['content'][:100]}...")
print(f"  Data source: {example['data_source']}")
print(f"  Ground truth: {example['reward_model']}")

## 2. Configuration

CAPO experiments use Hydra for configuration. The key configs are:

- `capo_trainer.yaml`: Base configuration extending VERL's PPO trainer
- `smoke_test.yaml`: Minimal config for testing

### Key Configuration Options

| Option | Description |
|--------|-------------|
| `algorithm.adv_estimator` | Advantage estimator: `capo`, `capo_eb_lite`, `capo_eb` |
| `capo.eb_lite.*` | EB-lite hyperparameters |
| `capo.eb_full.*` | Full EB hyperparameters |
| `trainer.total_training_steps` | Number of PPO update steps |
| `data.train_batch_size` | Global batch size for training |

In [ ]:
# View the CAPO-specific configuration
from pathlib import Path
import yaml

# Find config path
import capo.experiments.recipe.capo.config as config_module
config_dir = Path(config_module.__file__).parent

# Load and display CAPO trainer config
with open(config_dir / "capo_trainer.yaml") as f:
    config = yaml.safe_load(f)

print("CAPO Configuration Block:")
print(yaml.dump(config.get("capo", {}), default_flow_style=False))

## 3. Training Command

To run CAPO training, you use the main entrypoint with Hydra overrides.

### Minimal Smoke Test (CPU/single GPU)

```bash
# Prepare data first
python -m capo.experiments.scripts.data.prepare_countdown_dataset \
    --out_dir /tmp/countdown_data \
    --num_train 1000 \
    --num_test 100

# Run smoke test
python -m capo.experiments.recipe.capo.main_capo \
    --config-name smoke_test \
    actor_rollout_ref.model.path=Qwen/Qwen2-0.5B-Instruct \
    data.train_files=/tmp/countdown_data/train.parquet \
    data.val_files=/tmp/countdown_data/test.parquet
```

### Full Experiment (multi-GPU)

```bash
# Use the experiment scripts
cd src/capo/experiments/recipe/capo/scripts
bash E1_main_comparison.sh \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --data_dir /path/to/countdown_data \
    --gpus_per_node 8 \
    --steps 500
```

In [ ]:
# Generate the training command for our prepared data
train_file = DATA_DIR / "train.parquet"
val_file = DATA_DIR / "test.parquet"

cmd = f"""
python -m capo.experiments.recipe.capo.main_capo \\
    --config-name smoke_test \\
    actor_rollout_ref.model.path=Qwen/Qwen2-0.5B-Instruct \\
    data.train_files={train_file} \\
    data.val_files={val_file}
""".strip()

print("Training command:")
print(cmd)

## 4. Understanding CAPO Advantage Estimation

During training, CAPO computes advantages using one of three methods:

### `capo` (Basic)
Simple CAPO weighting without empirical Bayes smoothing.

### `capo_eb_lite` (Recommended for most cases)
EB-lite: Fast, closed-form empirical Bayes estimation.
- Estimates $\beta$ (shrinkage) via fixed-point iteration
- Good balance of speed and accuracy

### `capo_eb` (Full EB)
Full empirical Bayes with joint optimization of:
- $\beta$ (shrinkage strength)
- $\rho$ (autocorrelation)
- $\eta$ (noise scale)

More accurate but slower. Best for final experiments.

In [ ]:
# Demonstrate EB-lite on synthetic data
import torch
from capo.eb_core import eb_lite

# Simulate rewards from a single prompt with 8 responses
torch.manual_seed(42)
n_responses = 8
seq_lens = torch.tensor([50, 75, 100, 60, 80, 90, 70, 55])  # varying lengths

# Simulate per-token "rewards" (in practice these are advantages from rollouts)
# True signal + correlated noise
true_signal = torch.randn(n_responses)
per_token_rewards = []
for i, L in enumerate(seq_lens):
    noise = 0.5 * torch.randn(L.item())
    # AR(1) correlation in the noise
    for t in range(1, L.item()):
        noise[t] = 0.3 * noise[t-1] + noise[t]
    per_token_rewards.append(true_signal[i] + noise)

print(f"Simulated {n_responses} responses with lengths: {seq_lens.tolist()}")

In [ ]:
# Run EB-lite to estimate smoothed advantages
# Pad sequences for batching
max_len = seq_lens.max().item()
rewards_padded = torch.zeros(n_responses, max_len)
mask = torch.zeros(n_responses, max_len, dtype=torch.bool)

for i, (r, L) in enumerate(zip(per_token_rewards, seq_lens)):
    rewards_padded[i, :L] = r
    mask[i, :L] = True

# EB-lite estimation
result = eb_lite(
    rewards=rewards_padded,  # (N, T)
    mask=mask,  # (N, T)
    max_iters=20,
    tol=1e-4,
)

print(f"Estimated beta (shrinkage): {result['beta']:.4f}")
print(f"Converged in {result['iters']} iterations")

In [ ]:
# Compare raw vs smoothed advantages
raw_advantages = torch.tensor([r.mean().item() for r in per_token_rewards])
smoothed_advantages = result['advantages']  # EB-lite smoothed

print("\nComparison of advantage estimates:")
print(f"{'Response':<10} {'Raw':>10} {'EB-Smoothed':>12} {'True Signal':>12}")
print("-" * 46)
for i in range(n_responses):
    print(f"{i:<10} {raw_advantages[i]:>10.4f} {smoothed_advantages[i]:>12.4f} {true_signal[i]:>12.4f}")

# Correlation with true signal
raw_corr = torch.corrcoef(torch.stack([raw_advantages, true_signal]))[0, 1]
smooth_corr = torch.corrcoef(torch.stack([smoothed_advantages, true_signal]))[0, 1]
print(f"\nCorrelation with true signal:")
print(f"  Raw:      {raw_corr:.4f}")
print(f"  Smoothed: {smooth_corr:.4f}")

## 5. Monitoring Training

During training, CAPO logs metrics to:

1. **Console**: Real-time progress
2. **metrics.jsonl**: Line-delimited JSON for analysis
3. **WandB** (optional): If `WANDB_API_KEY` is set

Key metrics to watch:

| Metric | Description |
|--------|-------------|
| `train/reward_mean` | Average reward across batch |
| `train/policy_loss` | Policy gradient loss |
| `val/accuracy` | Validation accuracy (task-specific) |
| `capo/beta` | EB shrinkage parameter (if using EB) |

In [ ]:
# Example: parsing metrics.jsonl after a run
import json
from pathlib import Path

def parse_metrics(metrics_path: Path) -> list:
    """Parse JSONL metrics file."""
    records = []
    with open(metrics_path) as f:
        for line in f:
            records.append(json.loads(line))
    return records

# Demo with synthetic metrics
demo_metrics = [
    {"step": 0, "train/reward_mean": 0.1, "val/accuracy": 0.05},
    {"step": 50, "train/reward_mean": 0.3, "val/accuracy": 0.15},
    {"step": 100, "train/reward_mean": 0.5, "val/accuracy": 0.28},
    {"step": 150, "train/reward_mean": 0.6, "val/accuracy": 0.35},
]

print("Sample metrics progression:")
for m in demo_metrics:
    print(f"  Step {m['step']:>3}: reward={m['train/reward_mean']:.2f}, accuracy={m['val/accuracy']:.2f}")

## Summary

This tutorial covered:

1. **Data Preparation**: Creating VERL-compatible parquet datasets
2. **Configuration**: Understanding CAPO Hydra configs
3. **Training**: Commands for smoke tests and full experiments
4. **EB Estimation**: How CAPO computes smoothed advantages
5. **Monitoring**: Tracking training progress

### Next Steps

- Try running a smoke test with a small model
- Experiment with different `algorithm.adv_estimator` settings
- Use the analysis scripts to generate paper artifacts

In [ ]:
# Cleanup
import shutil
shutil.rmtree(DATA_DIR, ignore_errors=True)
print("Tutorial complete!")